# 🚀 LLM-Model: Master Colab Runner (`main.ipynb`)
File notebook này là **bảng điều khiển duy nhất** để chạy, theo dõi và so sánh các kiến trúc mô hình LLM trên Google Colab.

Tất cả các chức năng đã được module hóa:
- `data/prepare_data.py`: Sinh dữ liệu 2 giai đoạn theo `TRAINING_METHODOLOGY.md`
- `tokenizer/train_tokenizer.py`: Huấn luyện Byte-level BPE Tokenizer (chuẩn LLaMA / GPT-4)
- `data/process_data.py`: Đóng gói dữ liệu nhị phân uint16 tối ưu RAM & tốc độ đọc GPU
- `src/model.py` & `src/config.py`: Kiến trúc Decoder-only (RMSNorm, RoPE, Attention, SwiGLU)
- `train.py`: Huấn luyện 2 giai đoạn (80% Nền tảng + 20% Annealing) & thu thập toàn bộ metrics
- `generate.py`: Thử nghiệm sinh văn bản, viết code và giải toán suy luận CoT (`<think>`)
- `compare_models.py`: So sánh hiệu năng nhiều kiến trúc (Loss, Perplexity, GradNorm, Tốc độ)


## 1. Thiết lập Môi trường & Kiểm tra GPU


In [ ]:
# 0. Tự động tải repository nếu đang chạy trên Google Colab
import os
if not os.path.exists('src'):
    !git clone -b main https://github.com/khangkaka066/Alba-model.git
    %cd Alba-model

# 1. Kiểm tra GPU CUDA và cài đặt thư viện cần thiết
!nvidia-smi
!pip install -q tokenizers matplotlib

import torch
print("✅ PyTorch Version:", torch.__version__)
print("🔥 CUDA GPU Khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("🚀 Thiết bị GPU:", torch.cuda.get_device_name(0))

## 2. Bước 1: Tạo Dữ liệu 2 Giai đoạn (`data/prepare_data.py`)
Tuân thủ đúng chiến lược trong `TRAINING_METHODOLOGY.md`:
- **Phase 1 (Nền tảng):** 45% Edu Text, 35% Code, 10% Math, 10% General Knowledge
- **Phase 2 (Annealing):** 40% Toán & Logic, 40% Code sạch, 20% Synthetic CoT


In [ ]:
!python data/prepare_data.py

## 3. Bước 2: Huấn luyện Byte-level BPE Tokenizer (`tokenizer/train_tokenizer.py`)
- Khởi tạo từ 256 bytes UTF-8 cơ sở (không bao giờ lỗi `<unk>`)
- Tách riêng từng chữ số `0-9` độc lập (giúp làm toán cột dọc)
- Tích hợp các special tokens: `<pad>`, `<unk>`, `<bos>`, `<eos>`, `<think>`, `</think>`


In [ ]:
!python tokenizer/train_tokenizer.py

## 4. Bước 3: Đóng gói Nhị phân uint16 (`data/process_data.py`)
Mã hóa văn bản thành các file tensor nhị phân (`phase1_train.bin`, `phase1_val.bin`, `phase2_train.bin`, `phase2_val.bin`) để nạp trực tiếp qua GPU.


In [ ]:
!python data/process_data.py

## 5. Bước 4: Huấn luyện Mô hình 2 Giai đoạn (`train.py`)
- **80% số bước đầu (1 ➔ 2.000):** Huấn luyện nền tảng trên Phase 1 với Cosine LR schedule + Warmup
- **20% số bước cuối (2.001 ➔ 2.500):** Annealing Phase trên Phase 2 với LR hạ nhiệt dần về sát 0
- **Tự động lưu:** `checkpoints/best_model.pt`, `checkpoints/metrics.json` và `checkpoints/training_dashboard.png`


In [ ]:
!python train.py --exp_name modern_transformer_llama_26M --dim 512 --n_layers 8 --n_heads 8 --max_iters 2500 --batch_size 32 --seq_len 128 --lr 1e-3 --eval_interval 250

## 6. Hiển thị Dashboard 6 Biểu đồ Đo lường
Các biểu đồ được vẽ và lưu tự động gồm:
1. **Cross-Entropy Loss (Train vs Val)**
2. **Perplexity (PPL)**: Độ bất định từ vựng (càng thấp mô hình càng thông minh)
3. **Learning Rate Schedule**: Lộ trình thay đổi tốc độ học
4. **Gradient Norm**: Độ ổn định huấn luyện (phát hiện nổ / biến mất gradient)
5. **Throughput (Tokens / sec)**: Tốc độ xử lý của mô hình
6. **Bảng tóm tắt thông số và kết quả tốt nhất**


In [ ]:
from IPython.display import Image, display

# Hiển thị trực tiếp ảnh dashboard đã lưu
dashboard_path = "checkpoints/training_dashboard.png"
display(Image(dashboard_path))

## 7. Bước 5: Thử nghiệm Sinh văn bản, Viết Code & Giải toán (`generate.py`)


In [ ]:
# Thử nghiệm 1: Giải toán đố suy luận từng bước qua thẻ <think>
!python generate.py --prompt "Instruction: Solve the multi-step word problem with detailed logical chain of thought.\nQuestion: Alice has 35 apples. Alice gives 12 apples to Bob and buys 8 more apples. How many apples does Alice have now?\n<think>" --temperature 0.3

In [ ]:
# Thử nghiệm 2: Viết Code Python sạch kèm docstring và type annotations
!python generate.py --prompt "# Python Implementation: binary_search\ndef binary_search(arr: list[int], target: int) -> int:" --temperature 0.2

In [ ]:
# Thử nghiệm 3: Kiến thức tổng quát / Khoa học
!python generate.py --prompt "Topic: The Water Cycle\nExplanation:" --temperature 0.5

## 8. Nghiên cứu & So sánh Nhiều Kiến trúc Mô hình (`compare_models.py`)
Khi bạn thay đổi kiến trúc trong `src/model.py` (ví dụ: thử nghiệm LayerNorm thay vì RMSNorm, hoặc GELU thay vì SwiGLU), bạn chỉ cần:
1. Đặt tên thí nghiệm mới khi train: `!python train.py --exp_name my_new_architecture ...`
2. Chạy lệnh so sánh đối chiếu bên dưới để xuất biểu đồ so sánh:


In [ ]:
import glob
from IPython.display import Image, display

# Tự động phát hiện các mô hình đã huấn luyện trong thư mục checkpoints
metric_files = sorted(glob.glob('checkpoints/*_metrics.json'))
print('Các file metrics tìm thấy:', metric_files)
if len(metric_files) >= 2:
    files_arg = ' '.join(metric_files)
    !python compare_models.py --files {files_arg} --output comparison_result.png
    display(Image('comparison_result.png'))
else:
    print(f'Hiện có {len(metric_files)} mô hình. Khi có từ 2 mô hình trở lên, biểu đồ đối đầu sẽ tự động hiển thị tại đây!')